# NHANES Machine Learning Modeling

This notebook develops and compares machine-learning models to predict elevated HbA1c among U.S. adults without previously reported diabetes.

The prepared analysis dataset was created in `nhanes_data_preparation.ipynb`, and exploratory analysis was completed in `nhanes_eda.ipynb`.

#### Refreshing The Research Questions:

**RQ1.** Can non-laboratory demographic, anthropometric, behavioral, cardiovascular, and socioeconomic characteristics be used to identify elevated HbA1c among adults without a previously reported diabetes diagnosis?

**RQ2.** Which non-laboratory characteristics are most important for identifying elevated HbA1c?

**RQ3.** Do behavioral and socioeconomic characteristics improve the prediction of elevated HbA1c beyond basic factors such as age and BMI?

**RQ4.** Can elevated HbA1c be identified among adults who are not obese?


| Model | Purpose in your project |
|---|---|
| Dummy Classifier | Simple reference baseline |
| Logistic Regression | Interpretable statistical/ML baseline |
| Random Forest | Captures nonlinear relationships and interactions |
| Gradient Boosting / XGBoost | Stronger predictive model for structured/tabular data |

#### Loading data


In [144]:
from pathlib import Path

import pandas as pd
import numpy as np

data_path = Path("../data/nhanes_analysis_ready.parquet")

print("Data file:", data_path)
print("File exists:", data_path.exists())

Data file: ..\data\nhanes_analysis_ready.parquet
File exists: True


In [146]:
analysis_data = pd.read_parquet(data_path)
analysis_data.head()

,SEQN,age,sex,race_ethnicity,education,income_poverty_ratio,mec_weight,psu,stratum,cycle,...,vigorous_minutes_week,moderate_minutes_week,recreational_activity_minutes_week,smoked_100_cigarettes,current_smoking,sleep_hours,hypertension,high_cholesterol,elevated_hba1c,smoking_status
0,62161.0,22.0,1.0,3.0,3.0,3.15,104236.582554,1.0,91.0,2011-2012,...,0.0,0.0,0.0,2.0,NaN,8.0,2.0,2.0,0,Never
1,62164.0,44.0,2.0,3.0,4.0,1.67,127965.226204,1.0,94.0,2011-2012,...,300.0,45.0,345.0,2.0,NaN,8.0,2.0,2.0,0,Never
2,62169.0,21.0,1.0,6.0,3.0,0.33,14783.600953,1.0,92.0,2011-2012,...,0.0,0.0,0.0,2.0,NaN,8.0,2.0,2.0,0,Never
3,62172.0,43.0,2.0,4.0,3.0,2.02,27122.911908,2.0,96.0,2011-2012,...,0.0,0.0,0.0,1.0,1.0,8.0,2.0,2.0,0,Current
4,62174.0,80.0,1.0,3.0,5.0,4.30,27335.895242,3.0,90.0,2011-2012,...,0.0,0.0,0.0,2.0,NaN,9.0,1.0,1.0,0,Never


In [148]:
print("Rows:", analysis_data.shape[0])
print("Columns:", analysis_data.shape[1])

print("Unique SEQN:", analysis_data["SEQN"].nunique())
print("Duplicate SEQN:", analysis_data["SEQN"].duplicated().sum())

Rows: 17156
Columns: 32
Unique SEQN: 17156
Duplicate SEQN: 0


#### Defining Target and Predictors
The machine-learning target is `elevated_hba1c`, where:

- `0` = HbA1c below 5.7%
- `1` = HbA1c at or above 5.7%

Two predictor sets are created for comparison:

1. **Baseline model:** uses only age and BMI.
2. **Expanded model:** adds demographic, socioeconomic, anthropometric, behavioral, and health-history characteristics.

In [151]:
# Defining the binary target variable
target = "elevated_hba1c"


# Baseline predictor set
# This simple model uses only age and BMI.
# It will serve as a reference for evaluating whether
# additional predictors improve model performance.
baseline_features = [
    "age",
    "bmi"
]


# Expanded predictor set
# These predictors were selected based on the EDA,
# research questions, missingness, and redundancy analysis.
expanded_features = [
    "age",
    "income_poverty_ratio",
    "bmi",
    "waist_cm",
    "recreational_activity_minutes_week",
    "sedentary_minutes",
    "sleep_hours",
    "sex",
    "race_ethnicity",
    "education",
    "smoking_status",
    "hypertension",
    "high_cholesterol"
]


# Display the predictor sets
print("Target variable:")
print(target)

print("\nBaseline predictors:")
for feature in baseline_features:
    print("-", feature)

print("\nExpanded predictors:")
for feature in expanded_features:
    print("-", feature)

Target variable:
elevated_hba1c

Baseline predictors:
- age
- bmi

Expanded predictors:
- age
- income_poverty_ratio
- bmi
- waist_cm
- recreational_activity_minutes_week
- sedentary_minutes
- sleep_hours
- sex
- race_ethnicity
- education
- smoking_status
- hypertension
- high_cholesterol


**Target vector**

In [164]:
#target vector
y = analysis_data[target].copy()


#the baseline feature matrix
X_baseline = analysis_data[
    baseline_features
].copy()


#the expanded feature matrix
X_expanded = analysis_data[
    expanded_features
].copy()


#shapes of the modeling datasets
print("Target shape:", y.shape)

print(
    "Baseline feature shape:",
    X_baseline.shape
)

print(
    "Expanded feature shape:",
    X_expanded.shape
)

Target shape: (17156,)
Baseline feature shape: (17156, 2)
Expanded feature shape: (17156, 13)


In [166]:
#distribution of the target classes

target_distribution = (
    y.value_counts()
    .sort_index()
    .to_frame("count")
)

target_distribution["percent"] = (
    target_distribution["count"]
    / len(y)
    * 100
).round(2)

target_distribution.index = [
    "Normal HbA1c",
    "Elevated HbA1c"
]

target_distribution

,count,percent
Normal HbA1c,11693,68.16
Elevated HbA1c,5463,31.84


The target classes are moderately imbalanced, with approximately 68% of participants in the normal HbA1c group and 32% in the elevated HbA1c group.

Because of this imbalance, model performance will not be evaluated using accuracy alone. Recall, precision, F1-score, ROC-AUC, and Precision-Recall AUC will also be considered.

#### Train/test splitting

- **80% training data** will be used to train the machine-learning models.
- **20% testing data** will be held aside for final model evaluation.

A stratified split is used so that the proportion of normal and elevated HbA1c cases remains approximately the same in both datasets.

The same training and testing participants will be used for both the baseline and expanded models to ensure a fair comparison.

In [170]:
from sklearn.model_selection import train_test_split

# random_state=42 to make the split reproducible.
#splitting the expanded dataset

X_train, X_test, y_train, y_test = train_test_split(
    X_expanded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [172]:
#baseline training and testing datasets
#using the same participants from the expanded split.

X_train_baseline = X_train[
    baseline_features
].copy()

X_test_baseline = X_test[
    baseline_features
].copy()


# Renaming the expanded datasets for clarity
X_train_expanded = X_train.copy()
X_test_expanded = X_test.copy()

In [174]:
# Verifying the number of observations in each dataset

print("Total participants:", len(analysis_data))

print("\nTraining participants:")
print(len(y_train))

print("\nTesting participants:")
print(len(y_test))

print("\nBaseline training shape:")
print(X_train_baseline.shape)

print("\nBaseline testing shape:")
print(X_test_baseline.shape)

print("\nExpanded training shape:")
print(X_train_expanded.shape)

print("\nExpanded testing shape:")
print(X_test_expanded.shape)

Total participants: 17156

Training participants:
13724

Testing participants:
3432

Baseline training shape:
(13724, 2)

Baseline testing shape:
(3432, 2)

Expanded training shape:
(13724, 13)

Expanded testing shape:
(3432, 13)


In [176]:
# Comparing the percentage of elevated HbA1c cases
# in the training and testing datasets.

print("Training target distribution:")
print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Training target distribution:
elevated_hba1c
0    68.16
1    31.84
Name: proportion, dtype: float64

Testing target distribution:
elevated_hba1c
0    68.15
1    31.85
Name: proportion, dtype: float64


The dataset was divided into 80% training data and 20% testing data using stratified sampling.

Stratification preserved the distribution of normal and elevated HbA1c cases across both datasets.

The same participants are used in the baseline and expanded feature sets so that differences in model performance reflect the predictors and modeling approach rather than differences in the train/test samples.

All preprocessing steps, including missing-value imputation, categorical encoding, and numerical scaling, will be fitted using the training data only to prevent data leakage.

#### Preprocessing Pipeline

**Numerical predictors**
- Missing values will be replaced using the median of the training data.
- Numerical variables will be standardized using StandardScaler.

**Categorical predictors**
- Missing values will be replaced using the most frequent category in the training data.
- Categories will be converted into machine-readable dummy variables using one-hot encoding.

All preprocessing steps will be fitted using the training data only to prevent information from the test set from leaking into model development.

In [180]:
# Import preprocessing tools from scikit-learn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Predictor Types

The expanded feature set contains both numerical and categorical predictors.

Although several NHANES categorical variables are stored using numeric codes, they represent categories rather than continuous quantities and therefore require categorical encoding.

In [183]:
# Numerical predictors in the expanded feature set

expanded_numeric_features = [
    "age",
    "income_poverty_ratio",
    "bmi",
    "waist_cm",
    "recreational_activity_minutes_week",
    "sedentary_minutes",
    "sleep_hours"
]


# Categorical predictors in the expanded feature set

expanded_categorical_features = [
    "sex",
    "race_ethnicity",
    "education",
    "smoking_status",
    "hypertension",
    "high_cholesterol"
]


# Baseline predictors are both numerical

baseline_numeric_features = [
    "age",
    "bmi"
]

In [185]:
# Convert categorical predictors to object dtype
# and replace pandas <NA> values with np.nan so
# scikit-learn can process them consistently.

for column in expanded_categorical_features:

    X_train_expanded[column] = (
        X_train_expanded[column]
        .astype("object")
        .where(
            X_train_expanded[column].notna(),
            np.nan
        )
    )

    X_test_expanded[column] = (
        X_test_expanded[column]
        .astype("object")
        .where(
            X_test_expanded[column].notna(),
            np.nan
        )
    )

In [187]:
# Verify categorical data types

X_train_expanded[
    expanded_categorical_features
].dtypes

sex                 object
race_ethnicity      object
education           object
smoking_status      object
hypertension        object
high_cholesterol    object
dtype: object

In [189]:
# Checking the number of predictors in each group

print(
    "Expanded numerical predictors:",
    len(expanded_numeric_features)
)

print(
    "Expanded categorical predictors:",
    len(expanded_categorical_features)
)

print(
    "Total expanded predictors:",
    len(expanded_numeric_features)
    + len(expanded_categorical_features)
)

print(
    "Baseline predictors:",
    len(baseline_numeric_features)
)

Expanded numerical predictors: 7
Expanded categorical predictors: 6
Total expanded predictors: 13
Baseline predictors: 2


In [191]:
# Examining missing values in the expanded training dataset

training_missingness = pd.DataFrame({
    "missing_count":
        X_train_expanded.isna().sum(),

    "missing_percent":
        (
            X_train_expanded.isna().mean()
            * 100
        ).round(2)
})

training_missingness = (
    training_missingness
    .sort_values(
        "missing_percent",
        ascending=False
    )
)

training_missingness

,missing_count,missing_percent
income_poverty_ratio,1294,9.43
waist_cm,644,4.69
bmi,163,1.19
sedentary_minutes,80,0.58
high_cholesterol,63,0.46
sleep_hours,44,0.32
recreational_activity_minutes_week,17,0.12
hypertension,15,0.11
education,11,0.08
smoking_status,10,0.07


#### Numerical preprocessing

In [194]:
# Numerical preprocessing :
# 1. Replace missing numerical values with the training median
# 2. Standardize numerical variables

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

#### Categorical preprocessing

In [197]:
# Categorical preprocessing:
# 1. Replace missing categories with the most frequent category
# 2. Convert categories to one-hot encoded variables

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

**Combining those two pipelines for expanded preprocessing pipeline**

In [200]:
# Applying numerical preprocessing to numerical variables
# and categorical preprocessing to categorical variables

expanded_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            expanded_numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            expanded_categorical_features
        )
    ]
)

#### Baseline preprocessing pipeline

In [203]:
# Baseline preprocessing:
# Both age and BMI are numerical variables.

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            baseline_numeric_features
        )
    ]
)
print("Baseline preprocessing pipeline:")
print(baseline_preprocessor)

print("\nExpanded preprocessing pipeline:")
print(expanded_preprocessor)

Baseline preprocessing pipeline:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'bmi'])])

Expanded preprocessing pipeline:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'income_poverty_ratio', 'bmi',
                                  'waist_cm',
                                  'recreational_activity_minutes_week',
                                  'sedentary_minutes', 'sleep_hours']),
                                ('categorical',
                              

The preprocessing objects have been defined but will be incorporated directly into each machine-learning pipeline. This ensures that preprocessing is fitted only on the training data and is applied consistently to the test data.

### Reference Model: Dummy Classifier
A Dummy Classifier is used as a simple reference model. It predicts the majority class and therefore does not learn from the predictors.

This benchmark helps show whether the machine-learning models provide meaningful improvement beyond simply predicting the most common outcome.

In [207]:
# Import the reference classifier
from sklearn.dummy import DummyClassifier

# Import classification evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [209]:
# a Dummy Classifier that always predicts
# the most frequent class observed in the training data, 
# learns which target class occurs most often in y_train

dummy_model = DummyClassifier(
    strategy="most_frequent"
)


# Train the reference model using the training data
dummy_model.fit(
    X_train_baseline,
    y_train
)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.68,0.32]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](2,)","['age','bmi']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,2
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [211]:
# Predict the target class for the held-out test set

dummy_predictions = dummy_model.predict(
    X_test_baseline
)

# Obtain predicted probabilities for the positive class
# 1 = Elevated HbA1c

dummy_probabilities = (
    dummy_model.predict_proba(
        X_test_baseline
    )[:, 1]
)

In [213]:
# Calculate evaluation metrics for the Dummy Classifier

dummy_accuracy = accuracy_score(
    y_test,
    dummy_predictions
)

dummy_precision = precision_score(
    y_test,
    dummy_predictions,
    zero_division=0
)

dummy_recall = recall_score(
    y_test,
    dummy_predictions
)

dummy_f1 = f1_score(
    y_test,
    dummy_predictions
)

dummy_roc_auc = roc_auc_score(
    y_test,
    dummy_probabilities
)

dummy_pr_auc = average_precision_score(
    y_test,
    dummy_probabilities
)

In [215]:
#reference-model performance in a table

dummy_results = pd.DataFrame({
    "Model": [
        "Dummy Classifier"
    ],
    "Accuracy": [
        dummy_accuracy
    ],
    "Precision": [
        dummy_precision
    ],
    "Recall": [
        dummy_recall
    ],
    "F1": [
        dummy_f1
    ],
    "ROC_AUC": [
        dummy_roc_auc
    ],
    "PR_AUC": [
        dummy_pr_auc
    ]
})

dummy_results.round(3)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Dummy Classifier,0.682,0.0,0.0,0.0,0.5,0.318


In [217]:
# Convert the confusion matrix into a labeled table

dummy_cm_table = pd.DataFrame(
    dummy_confusion_matrix,
    index=[
        "Actual Normal",
        "Actual Elevated"
    ],
    columns=[
        "Predicted Normal",
        "Predicted Elevated"
    ]
)

dummy_cm_table

,Predicted Normal,Predicted Elevated
Actual Normal,2339,0
Actual Elevated,1093,0


Decent accuracy while identifying zero elevated-HbA1c participants

In [220]:
# precision, recall, and F1-score
# separately for each target class

print(
    classification_report(
        y_test,
        dummy_predictions,
        target_names=[
            "Normal HbA1c",
            "Elevated HbA1c"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

  Normal HbA1c       0.68      1.00      0.81      2339
Elevated HbA1c       0.00      0.00      0.00      1093

      accuracy                           0.68      3432
     macro avg       0.34      0.50      0.41      3432
  weighted avg       0.46      0.68      0.55      3432



In [222]:
# Counts of the predictions made by the Dummy Classifier

pd.Series(
    dummy_predictions
).value_counts().sort_index()

0    3432
Name: count, dtype: int64

The Dummy Classifier achieved approximately 68% accuracy but 0% recall for elevated HbA1c because it predicted every participant as belonging to the majority class. This demonstrates why accuracy alone is not sufficient for evaluating the predictive models.

###  Logistic Regression
Logistic Regression is used as the first predictive model because the outcome, `elevated_hba1c`, is binary.

The model estimates the probability that a participant has elevated HbA1c based on the selected non-laboratory predictors.

Two Logistic Regression models are compared:

1. **Baseline model:** age and BMI only
2. **Expanded model:** demographic, socioeconomic, anthropometric, behavioral, and health-history predictors

This comparison will help determine whether the expanded predictor set improves prediction beyond age and BMI alone.

In [226]:
# Import Logistic Regression
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

**Baseline Logistic Regression: Age and BMI**

The baseline Logistic Regression model uses only age and BMI.

This provides a simple predictive benchmark and allows the expanded model to be evaluated based on whether additional non-laboratory characteristics improve predictive performance.

In [229]:
# modeling pipeline that combines:
# 1. Baseline preprocessing
# 2. Logistic Regression

baseline_logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            baseline_preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [231]:
# Train the baseline Logistic Regression model
# using the training data only

baseline_logistic_model.fit(
    X_train_baseline,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['age','bmi']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but p

In [233]:
# Predict the target class for the test set

baseline_logistic_predictions = (
    baseline_logistic_model.predict(
        X_test_baseline
    )
)


# Predict the probability of elevated HbA1c
# Class 1 = Elevated HbA1c

baseline_logistic_probabilities = (
    baseline_logistic_model.predict_proba(
        X_test_baseline
    )[:, 1]
)

In [235]:
# Calculate evaluation metrics

baseline_logistic_accuracy = accuracy_score(
    y_test,
    baseline_logistic_predictions
)

baseline_logistic_precision = precision_score(
    y_test,
    baseline_logistic_predictions
)

baseline_logistic_recall = recall_score(
    y_test,
    baseline_logistic_predictions
)

baseline_logistic_f1 = f1_score(
    y_test,
    baseline_logistic_predictions
)

baseline_logistic_roc_auc = roc_auc_score(
    y_test,
    baseline_logistic_probabilities
)

baseline_logistic_pr_auc = average_precision_score(
    y_test,
    baseline_logistic_probabilities
)

In [237]:
# Store baseline Logistic Regression performance

baseline_logistic_results = pd.DataFrame({
    "Model": [
        "Logistic Regression"
    ],
    "Feature_Set": [
        "Age + BMI"
    ],
    "Accuracy": [
        baseline_logistic_accuracy
    ],
    "Precision": [
        baseline_logistic_precision
    ],
    "Recall": [
        baseline_logistic_recall
    ],
    "F1": [
        baseline_logistic_f1
    ],
    "ROC_AUC": [
        baseline_logistic_roc_auc
    ],
    "PR_AUC": [
        baseline_logistic_pr_auc
    ]
})

baseline_logistic_results.round(3)

,Model,Feature_Set,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,Age + BMI,0.704,0.558,0.345,0.426,0.752,0.536


In [239]:
# Creating a labeled confusion matrix

baseline_logistic_cm = confusion_matrix(
    y_test,
    baseline_logistic_predictions
)

baseline_logistic_cm_table = pd.DataFrame(
    baseline_logistic_cm,
    index=[
        "Actual Normal",
        "Actual Elevated"
    ],
    columns=[
        "Predicted Normal",
        "Predicted Elevated"
    ]
)

baseline_logistic_cm_table

,Predicted Normal,Predicted Elevated
Actual Normal,2040,299
Actual Elevated,716,377


In [241]:
print(
    classification_report(
        y_test,
        baseline_logistic_predictions,
        target_names=[
            "Normal HbA1c",
            "Elevated HbA1c"
        ]
    )
)

                precision    recall  f1-score   support

  Normal HbA1c       0.74      0.87      0.80      2339
Elevated HbA1c       0.56      0.34      0.43      1093

      accuracy                           0.70      3432
     macro avg       0.65      0.61      0.61      3432
  weighted avg       0.68      0.70      0.68      3432



**Baseline Logistic Regression Findings**

The baseline Logistic Regression model using only age and BMI achieved an accuracy of 70.4% and a ROC-AUC of 0.752.

For elevated HbA1c, the model achieved a precision of 55.8%, recall of 34.5%, and F1-score of 0.426.

Of the 1,093 participants with elevated HbA1c in the test set, the model correctly identified 377 and missed 716.

Compared with the Dummy Classifier, the age-and-BMI model provides meaningful predictive discrimination. However, its relatively low recall indicates that age and BMI alone miss many participants with elevated HbA1c.

### Expanded Logistic Regression

The expanded Logistic Regression model includes the full set of selected non-laboratory predictors.

The goal is to determine whether demographic, socioeconomic, behavioral, anthropometric, and health-history characteristics improve prediction beyond age and BMI alone.

In [245]:
# Create the expanded Logistic Regression pipeline

expanded_logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            expanded_preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [247]:
# Train the expanded Logistic Regression model

expanded_logistic_model.fit(
    X_train_expanded,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['age','income_poverty_ratio','bmi',...,'smoking_status','hypertension', 'high_cholesterol']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specif

**Expanded Logistic Regression Predictions**

The trained expanded Logistic Regression model is applied to the held-out test set.

Two outputs are generated:

- Predicted class (`0` = Normal HbA1c, `1` = Elevated HbA1c)
- Predicted probability of elevated HbA1c

In [250]:
# Predict the HbA1c class for each participant
# in the held-out test dataset.

expanded_logistic_predictions = (
    expanded_logistic_model.predict(
        X_test_expanded
    )
)


# Predict the probability that each participant
# has elevated HbA1c (class = 1).

expanded_logistic_probabilities = (
    expanded_logistic_model.predict_proba(
        X_test_expanded
    )[:, 1]
)

In [252]:
print(
    "First 10 predicted classes:"
)

print(
    expanded_logistic_predictions[:10]
)

print(
    "\nFirst 10 predicted probabilities:"
)

print(
    np.round(
        expanded_logistic_probabilities[:10],
        3
    )
)

First 10 predicted classes:
[0 0 0 0 1 0 1 0 0 0]

First 10 predicted probabilities:
[0.291 0.065 0.127 0.222 0.541 0.259 0.564 0.184 0.331 0.184]


**Expanded Logistic Regression Evaluation**

The expanded Logistic Regression model is evaluated on the held-out test set using accuracy, precision, recall, F1-score, ROC-AUC, and Precision-Recall AUC.

Because elevated HbA1c is the positive class, particular attention is given to recall and F1-score.

In [255]:
# Calculate evaluation metrics for the expanded Logistic Regression model

expanded_logistic_accuracy = accuracy_score(
    y_test,
    expanded_logistic_predictions
)

expanded_logistic_precision = precision_score(
    y_test,
    expanded_logistic_predictions
)

expanded_logistic_recall = recall_score(
    y_test,
    expanded_logistic_predictions
)

expanded_logistic_f1 = f1_score(
    y_test,
    expanded_logistic_predictions
)

expanded_logistic_roc_auc = roc_auc_score(
    y_test,
    expanded_logistic_probabilities
)

expanded_logistic_pr_auc = average_precision_score(
    y_test,
    expanded_logistic_probabilities
)

In [257]:
expanded_logistic_results = pd.DataFrame({
    "Model": [
        "Logistic Regression"
    ],
    "Feature_Set": [
        "Expanded"
    ],
    "Accuracy": [
        expanded_logistic_accuracy
    ],
    "Precision": [
        expanded_logistic_precision
    ],
    "Recall": [
        expanded_logistic_recall
    ],
    "F1": [
        expanded_logistic_f1
    ],
    "ROC_AUC": [
        expanded_logistic_roc_auc
    ],
    "PR_AUC": [
        expanded_logistic_pr_auc
    ]
})

expanded_logistic_results.round(3)

,Model,Feature_Set,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,Expanded,0.727,0.602,0.423,0.497,0.777,0.586


### Expanded Logistic Regression Findings

The expanded Logistic Regression model improved performance compared with the age-and-BMI baseline model.

Accuracy increased from 70.4% to 72.7%, while recall for elevated HbA1c increased from 34.5% to 42.3%. Precision improved from 55.8% to 60.2%, and the F1-score increased from 0.426 to 0.497.

Discrimination also improved, with ROC-AUC increasing from 0.752 to 0.777 and PR-AUC increasing from 0.536 to 0.586.

These results suggest that the additional demographic, socioeconomic, behavioral, anthropometric, and health-history predictors provide useful information beyond age and BMI alone.

In [260]:
# Creating the confusion matrix for the expanded Logistic Regression model

expanded_logistic_cm = confusion_matrix(
    y_test,
    expanded_logistic_predictions
)

expanded_logistic_cm_table = pd.DataFrame(
    expanded_logistic_cm,
    index=[
        "Actual Normal",
        "Actual Elevated"
    ],
    columns=[
        "Predicted Normal",
        "Predicted Elevated"
    ]
)

expanded_logistic_cm_table

,Predicted Normal,Predicted Elevated
Actual Normal,2034,305
Actual Elevated,631,462


In [262]:
# Display precision, recall, and F1-score
# for each HbA1c class

print(
    classification_report(
        y_test,
        expanded_logistic_predictions,
        target_names=[
            "Normal HbA1c",
            "Elevated HbA1c"
        ]
    )
)

                precision    recall  f1-score   support

  Normal HbA1c       0.76      0.87      0.81      2339
Elevated HbA1c       0.60      0.42      0.50      1093

      accuracy                           0.73      3432
     macro avg       0.68      0.65      0.65      3432
  weighted avg       0.71      0.73      0.71      3432



### Expanded Logistic Regression Performance

The expanded Logistic Regression model correctly identified 462 of the 1,093 participants with elevated HbA1c and missed 631.

For elevated HbA1c, the model achieved a precision of 60%, recall of 42%, and F1-score of 0.50. This means that approximately 42% of participants who truly had elevated HbA1c were identified by the model, and approximately 60% of participants predicted as elevated actually had elevated HbA1c.

Compared with the age-and-BMI baseline model, the expanded model identified 85 additional elevated-HbA1c cases while producing only a small increase in false-positive predictions.

Overall, the expanded predictor set improved accuracy, precision, recall, F1-score, ROC-AUC, and PR-AUC, suggesting that the additional demographic, socioeconomic, behavioral, anthropometric, and health-history characteristics provide predictive information beyond age and BMI alone.

In [ ]:
#### Random Forest
